In [ ]:
import tiktoken

# tokenizer 工具把文本转为token
enc = tiktoken.encoding_for_model("gpt-4o")
text = "我喜欢喝Coffee"
tokens = enc.encode(text)
print(tokens)  # 输出 token ID 列表, 模型用这些ID查表生成嵌入或预测


[7522, 69681, 109855, 90651]


In [1]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

client = OpenAI()

def get_embedding(text, model='text-embedding-v3'):
    response = client.embeddings.create(model=model, input=text)

    vector = response.data[0].embedding
    return vector


# 测试
text = "Limbo.py 也做AI"
vector = get_embedding(text)
print(f"文本: {text}")
print(f"向量维度: {len(vector)}")
print(f"前5维: {vector[:5]}")

文本: Limbo.py 也做AI
向量维度: 1024
前5维: [-0.05814730376005173, 0.01104332972317934, -0.030975665897130966, -0.000890962837729603, -0.06532837450504303]


In [ ]:
# 计算语义相似度
import numpy as np
import pandas as pd

def cosine_similarity(vec1, vec2):
    return np.dot(vec1, vec2) / (np.linalg.norm(vec1) * np.linalg.norm(vec2))

# 对比多组文本的相似度
texts = [
    "Limbo.py也做AI",
    "LLM",
    "limbo",
    "狗子",
    "人工智能"
]


# 获取所有文本的 embedding
embeddings = [get_embedding(t) for t in texts]


# 计算相似度矩阵
matrix = [[cosine_similarity(embeddings[i], embeddings[j]) for j in range(len(texts))] 
          for i in range(len(texts))]
df = pd.DataFrame(matrix, index=texts, columns=texts).round(3)

print("相似度矩阵:")
df

相似度矩阵:


,Limbo.py也做AI,LLM,limbo,狗子,人工智能
Limbo.py也做AI,1.000,0.500,0.646,0.446,0.660
LLM,0.500,1.000,0.597,0.403,0.504
limbo,0.646,0.597,1.000,0.516,0.485
狗子,0.446,0.403,0.516,1.000,0.465
人工智能,0.660,0.504,0.485,0.465,1.000


In [11]:
# Embedding 实际场景之语义搜索

class SemanticSearcher:
    """基于 Embedding 的语义搜索"""
    
    def __init__(self):
        self.documents = []      # 原始文档
        self.embeddings = []     # 文档 embedding
    
    def add_document(self, text):
        """添加文档到搜索库"""
        self.documents.append(text)
        self.embeddings.append(get_embedding(text))
    
    def search(self, query, top_k=3):
        """语义搜索"""
        query_vec = get_embedding(query)
        
        # 计算与所有文档的相似度
        similarities = [
            cosine_similarity(query_vec, doc_vec)
            for doc_vec in self.embeddings
        ]
        
        # 返回最相似的 top_k 个
        top_indices = np.argsort(similarities)[-top_k:][::-1]
        return [(self.documents[i], similarities[i]) for i in top_indices]

# 使用示例
searcher = SemanticSearcher()
docs = [
    "Python 是一种高级编程语言，语法简洁",
    "JavaScript 主要用于网页前端开发",
    "机器学习是人工智能的一个分支",
    "你用过PHP吗？",
]


for doc in docs:
    searcher.add_document(doc)

# 语义搜索：查询"AI编程"，应该匹配到 Python 和机器学习相关文档
results = searcher.search("AI 编程语言", top_k=2)
print("搜索结果:")
for doc, score in results:
    print(f"  [{score:.3f}] {doc}")

搜索结果:
  [0.645] Python 是一种高级编程语言，语法简洁
  [0.569] 机器学习是人工智能的一个分支


In [ ]:
# Embedding 实际场景之文本分类（zero shot）

def zero_shot_classify(text, categories):
    """零样本文本分类"""
    # 核心思想：不需要任何训练样本，直接利用 embedding 的语义对齐能力
    # 把待分类文本和每个类别名称都映射到同一个向量空间
    # "语义越接近的文本，向量距离越近" —— 这正是 embedding 模型的训练目标

    # 将待分类文本编码为向量
    text_vec = get_embedding(text)

    # 将每个类别名称也编码为向量
    # 类别名本身就携带语义，embedding 模型能理解"科技""体育"等词的含义
    category_vectors = [get_embedding(c) for c in categories]

    # 计算文本向量与每个类别向量的余弦相似度
    # 相似度越高 → 语义越近 → 越可能属于该类别
    similarities = [cosine_similarity(text_vec, cv) for cv in category_vectors]

    # 取相似度最高的类别作为分类结果
    best_idx = np.argmax(similarities)

    return categories[best_idx], similarities[best_idx]